In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

ROOT=Path.cwd().parents[1]
sys.path.insert(0,str(ROOT))

import common as cfg
import helper_funcs as hf

CONFIG=cfg.load_config()
INPUT,OUTPUT,SRC=cfg.task_paths()
OUTPUT.mkdir(parents=True,exist_ok=True)
rng=cfg.initialize(CONFIG['seed'])


In [ ]:
SOURCE=INPUT/'a_import'
bundle=hf.load_bundle(SOURCE)
raw_source=SOURCE/CONFIG['raw_trajectory_file']
raw_q=None
raw_ids=None
if raw_source.exists():
    raw_ids,raw_q=hf.read_trajectory_matrix(raw_source,CONFIG['n_years'])


In [ ]:
targets=hf.build_empirical_targets(bundle,raw_q)
followup=hf.followup_distribution(bundle['age_coverage'])
priors=hf.prior_table()
edges=hf.hypothesis_edges()
boundary=pd.DataFrame([
    {'item':'observable','value':'annual harvest count H_it'},
    {'item':'count_definition','value':'unweighted integer annual q'},
    {'item':'latent','value':'G, N, D, W, Z, K and project states'},
    {'item':'followup','value':'empirical age-coverage distribution'},
    {'item':'raw_trajectories_available','value':raw_q is not None},
    {'item':'annual_horizon','value':CONFIG['n_years']},
    {'item':'entry_condition','value':'latent project stock at first observed roster year'}
])


In [ ]:
targets.to_csv(OUTPUT/'empirical_targets.csv',index=False)
followup.to_csv(OUTPUT/'followup_distribution.csv',index=False)
priors.to_csv(OUTPUT/'prior_specification.csv',index=False)
edges.to_csv(OUTPUT/'hypothesis_edges.csv',index=False)
boundary.to_csv(OUTPUT/'observation_boundary.csv',index=False)
if raw_q is not None:
    raw=pd.DataFrame(raw_q,columns=[str(x) for x in range(CONFIG['n_years'])])
    raw.insert(0,'scientist_id',raw_ids)
    raw.to_csv(OUTPUT/'raw_trajectories.csv',index=False)


In [ ]:
fig,ax=plt.subplots(figsize=(8,4.5))
counts=targets.groupby('block',as_index=False).size()
sns.barplot(data=counts,y='block',x='size',ax=ax)
ax.set(xlabel='target metrics',ylabel='',title='ABC target blocks')
fig.tight_layout()
fig.savefig(OUTPUT/'target_blocks.png',dpi=220)
plt.close(fig)
targets.groupby('block').size()
